# Lets create fiass DB and try to retrieve data

In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import FAISS

c:\Generative AI\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
from langchain_ollama import OllamaEmbeddings

In [2]:
text_loader = TextLoader('speech.txt')

docs = text_loader.load()
docs

[Document(metadata={'source': 'speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.\n\nJust because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.\n\n…\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness be

In [5]:
# split the data

text_splitter = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=200,
    chunk_overlap=30
)

final_docs = text_splitter.split_documents(docs)
final_docs

Created a chunk of size 470, which is longer than the specified 200
Created a chunk of size 347, which is longer than the specified 200
Created a chunk of size 668, which is longer than the specified 200
Created a chunk of size 982, which is longer than the specified 200
Created a chunk of size 789, which is longer than the specified 200


[Document(metadata={'source': 'speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.'),
 Document(metadata={'source': 'speech.txt'}, page_content='Just because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.'),
 Document(metadata={'source': 'speech.txt'}, page_content

In [ ]:
# Lets embed the data | by the way this gemma:2b is not an ideal model for embedding this is a text generation model. im using this just for practice
embeddings = OllamaEmbeddings(
    model='gemma:2b'
)

# creat vector db
db = FAISS.from_documents(
    documents=final_docs,
    embedding=embeddings
)

db

C:\Users\hidel\AppData\Local\Temp\ipykernel_25960\3260512188.py:2: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(


In [7]:
# Lets query some data

query = "What does the speaker believe is the main reason the United States of America should enter the war?"

results = db.similarity_search(query=query)
results

[Document(id='9945bd87-13c8-4a9b-9670-3388ff4308c3', metadata={'source': 'speech.txt'}, page_content='Just because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.'),
 Document(id='dff85339-24fb-4a68-9f17-2b4eabd2230a', metadata={'source': 'speech.txt'}, page_content='It will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German 

In [8]:
results[0]

Document(id='9945bd87-13c8-4a9b-9670-3388ff4308c3', metadata={'source': 'speech.txt'}, page_content='Just because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.')

# Get data using Retriever

In [9]:
retriever = db.as_retriever()
retriever.invoke(query)

[Document(id='9945bd87-13c8-4a9b-9670-3388ff4308c3', metadata={'source': 'speech.txt'}, page_content='Just because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.'),
 Document(id='dff85339-24fb-4a68-9f17-2b4eabd2230a', metadata={'source': 'speech.txt'}, page_content='It will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German 

In [11]:
docs_and_score = db.similarity_search_with_score(query=query)
docs_and_score

[(Document(id='9945bd87-13c8-4a9b-9670-3388ff4308c3', metadata={'source': 'speech.txt'}, page_content='Just because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.'),
  np.float32(3442.1763)),
 (Document(id='dff85339-24fb-4a68-9f17-2b4eabd2230a', metadata={'source': 'speech.txt'}, page_content='It will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the si

In [12]:
# We can pass vectors too
embedding_vectors = embeddings.embed_query(query)
embedding_vectors

[-0.29368454217910767,
 1.4657374620437622,
 0.29028913378715515,
 0.8727983236312866,
 0.4449019134044647,
 0.7749341726303101,
 0.8761276602745056,
 0.5445429086685181,
 0.7611392140388489,
 0.2693311870098114,
 -1.3360661268234253,
 -0.13252666592597961,
 -0.2530159652233124,
 1.1539369821548462,
 1.3896502256393433,
 -0.44070661067962646,
 2.973386764526367,
 1.3588582277297974,
 1.5296578407287598,
 0.7879416942596436,
 0.9254499077796936,
 -0.7938965559005737,
 1.7958667278289795,
 0.7459902167320251,
 -0.4622628092765808,
 -1.0758024454116821,
 -1.7741186618804932,
 -1.8988834619522095,
 -1.0117019414901733,
 -2.1629629135131836,
 3.3995070457458496,
 -0.7976176738739014,
 1.297654628753662,
 -0.071863554418087,
 -0.8213942050933838,
 -0.4656428098678589,
 1.165165662765503,
 -1.0148718357086182,
 0.2180904895067215,
 -0.3568763732910156,
 -0.310542494058609,
 -0.28938618302345276,
 0.6985461711883545,
 -0.22150829434394836,
 -0.5985087752342224,
 0.7884124517440796,
 0.37972781

In [13]:
docs_score_vec = db.similarity_search_by_vector(embedding_vectors)
docs_score_vec

[Document(id='9945bd87-13c8-4a9b-9670-3388ff4308c3', metadata={'source': 'speech.txt'}, page_content='Just because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.'),
 Document(id='dff85339-24fb-4a68-9f17-2b4eabd2230a', metadata={'source': 'speech.txt'}, page_content='It will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German 

In [14]:
# Saving and loading
db.save_local("faiss_index")

In [17]:
# Load the db

new_db = FAISS.load_local('faiss_index', embeddings=embeddings, allow_dangerous_deserialization=True)


In [18]:
docs = new_db.similarity_search(query=query)